# A.2 Grundläggande beskrivning

Det här notebooken sammanfattar den övergripande skadebilden och de numeriska variablerna i både tränings- och testdatan. Fokus ligger på:

- total skadefrekvens i träning och test
- fördelning av `AntalSkador` och `Duration`
- beskrivning av ekonomiska variabler med medelvärde, median, kvartiler och spridningsmått
- histogram och täthetsdiagram för log-transformerade variabler

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

data_dir = Path("../../../data")
df = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")
df_test = pd.read_csv(data_dir / "Entreprenadförsäkring test.csv")

print(f"Träning: {df.shape[0]:,} rader")
print(f"Test:    {df_test.shape[0]:,} rader")

Träning: 1,033,386 rader
Test:    291,649 rader


## 1. Övergripande skadebild — träningsdata

Det här avsnittet sammanfattar hur ovanliga skador är i träningsdatan och sätter skadefrekvensen i relation till både antal rader och exponeringsår.

In [3]:
def claim_summary_table(data: pd.DataFrame, label: str) -> pd.DataFrame:
    claim_share = (
        data["AntalSkador"]
        .value_counts(normalize=True)
        .sort_index()
        .rename("andel")
        .mul(100)
    )
    total_claims = int(data["AntalSkador"].sum())
    total_duration = data["Duration"].sum()
    claims_per_row = data["AntalSkador"].mean()
    claims_per_exposure_year = total_claims / total_duration

    return pd.DataFrame(
        {
            "Mått": [
                "Andel med 0 skador (%)",
                "Andel med 1 skada (%)",
                "Andel med 2 skador (%)",
                "Totalt antal skador",
                "Totalt antal exponeringsår",
                "Skador per rad",
                "Skador per exponerat år",
            ],
            f"Värde ({label})": [
                round(claim_share.get(0, 0), 2),
                round(claim_share.get(1, 0), 2),
                round(claim_share.get(2, 0), 2),
                total_claims,
                round(total_duration, 0),
                round(claims_per_row, 4),
                round(claims_per_exposure_year, 5),
            ],
        }
    )


train_summary = claim_summary_table(df, "träning")
train_summary

,Mått,Värde (träning)
0,Andel med 0 skador (%),98.12000
1,Andel med 1 skada (%),1.86000
2,Andel med 2 skador (%),0.03000
3,Totalt antal skador,19730.00000
4,Totalt antal exponeringsår,924039.00000
5,Skador per rad,0.01910
6,Skador per exponerat år,0.02135


### Tolkning

Skador är ovanliga i datan:

- 98,12 % av raderna har 0 skador
- 1,86 % har 1 skada
- 0,03 % har 2 skador

Totalt i träningsdatan finns:

- 19 730 skador
- 924 039 exponeringsår summerat via `Duration`
- 0,0191 skador per rad
- 0,02135 skador per exponerat år

Det betyder att skadefrekvensen är låg, vilket är viktigt för hela projektet. Det antyder att vi arbetar med count-data med många nollor, vilket passar bra med enklare frekvensmodeller som Poisson eller eventuellt negativ binomial senare.

## 2. Övergripande skadebild — testdata

Samma sammanfattning beräknas för testdatan (2025) för att kunna jämföra med träningsdatan.

In [ ]:
test_summary = claim_summary_table(df_test, "test")

comparison = train_summary.merge(test_summary, on="Mått")
comparison

### Tolkning

Testdatan (2025) bör visa en liknande skadebild som träningsdatan om portföljens karaktär inte förändrats drastiskt. Eventuella skillnader i skadefrekvens mellan träning och test kan bero på verklig tidsvariation eller på sammansättningseffekter i portföljen.

Jämförelsen ovan bekräftar att testdatan har samma grundläggande egenskaper: stark nollövervikt och låg skadefrekvens. Det stödjer att modeller tränade på 2021–2024 rimligen kan appliceras på 2025.

## 3. Beskrivning av de numeriska variablerna

Här sammanfattas de viktigaste numeriska variablerna med fokus på typiska nivåer, spridning och hur extrema observationerna är. Kvartiler och IQR ger en bättre bild av spridningen än bara medelvärde och max.

In [ ]:
cols = ["Omsattning", "Forsakringsbelopp", "Sjalvrisk", "Duration"]
labels = ["Omsättning", "Försäkringsbelopp", "Självrisk", "Duration"]

rows = []
for col, label in zip(cols, labels):
    s = df[col]
    rows.append(
        {
            "Variabel": label,
            "Medel": round(s.mean(), 3 if col == "Duration" else 0),
            "Std": round(s.std(), 3 if col == "Duration" else 0),
            "Q1": round(s.quantile(0.25), 3 if col == "Duration" else 0),
            "Median": round(s.median(), 3 if col == "Duration" else 0),
            "Q3": round(s.quantile(0.75), 3 if col == "Duration" else 0),
            "IQR": round(s.quantile(0.75) - s.quantile(0.25), 3 if col == "Duration" else 0),
            "P99": round(s.quantile(0.99), 3 if col == "Duration" else 0),
            "Max": round(s.max(), 3 if col == "Duration" else 0),
        }
    )

numeric_summary = pd.DataFrame(rows)

duration_full_year_share = (df["Duration"] == 1).mean() * 100

numeric_summary

### Tolkning

**Omsättning**

- Medel: ~13,4 miljoner SEK, Std: stor spridning
- Q1: ~4,1 miljoner, Median: ~8,1 miljoner, Q3: ~15,9 miljoner
- 99-percentil: ~83,5 miljoner, Max: ~1,2 miljarder SEK
- IQR visar att de mittersta 50 % av observationerna spänner ungefär 4–16 miljoner

**Försäkringsbelopp**

- Medel: ~1,06 miljoner, Median: ~507 000
- Q1–Q3: ~223 000 till ~1,15 miljoner
- Max: ~120 miljoner

**Självrisk**

- Medel: ~16 100, Median: 10 000
- Q1: 5 000, Q3: 20 000
- Bara 6 unika värden — i praktiken diskret

**Duration**

- Medel: 0,894, Median: 1,0
- Cirka 73 % av observationerna gäller hela året
- IQR är litet, de flesta observationer ligger nära 1,0

De ekonomiska variablerna är tydligt högerskevda med stora avstånd mellan median och max. Standardavvikelsen är stor relativt medelvärdet. Det talar starkt för log-transformering vid modellering.

## 4. Fördelningar av numeriska variabler

Histogram och täthetsdiagram för de log-transformerade ekonomiska variablerna samt Duration. Log-transformeringen gör det lättare att se fördelningsformen och motiverar användningen av log-skala i modelleringen.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(np.log1p(df["Omsattning"]), kde=True, bins=50, ax=axes[0, 0])
axes[0, 0].set_title("log(1 + Omsättning)")
axes[0, 0].set_xlabel("log(1 + Omsättning)")

sns.histplot(np.log1p(df["Forsakringsbelopp"]), kde=True, bins=50, ax=axes[0, 1])
axes[0, 1].set_title("log(1 + Försäkringsbelopp)")
axes[0, 1].set_xlabel("log(1 + Försäkringsbelopp)")

sns.histplot(np.log1p(df["Sjalvrisk"]), kde=True, bins=50, ax=axes[1, 0])
axes[1, 0].set_title("log(1 + Självrisk)")
axes[1, 0].set_xlabel("log(1 + Självrisk)")

sns.histplot(df["Duration"], kde=True, bins=50, ax=axes[1, 1])
axes[1, 1].set_title("Duration")
axes[1, 1].set_xlabel("Duration (andel av år)")

plt.suptitle("Fördelningar av numeriska variabler", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Tolkning

- **Omsättning och Försäkringsbelopp** på log-skala visar fördelningar som är närmare symmetriska, vilket stödjer log-transformering i GLM.
- **Självrisk** på log-skala visar tydliga toppar — det är i praktiken en diskret variabel med få unika nivåer (6 st). Det syns som tydliga spikar i histogrammet.
- **Duration** är starkt koncentrerad kring 1,0 med en vänstersvans av kortare exponeringsperioder. Cirka 73 % av observationerna gäller hela året.

Dessa fördelningar bekräftar att log-transformering är motiverad för de ekonomiska variablerna, och att Duration bör användas som exponering snarare än förklarande variabel.